In [3]:
import pandas as pd
import numpy as np
import seaborn as sns

In [4]:
myData = pd.read_excel("Engsoccer2017-18.xlsx")
myData.head()

,Div,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR
0,EPL,2017-11-08 00:00:00,Arsenal,Leicester,4,3,H
1,EPL,2017-12-08 00:00:00,Brighton,Man City,0,2,A
2,EPL,2017-12-08 00:00:00,Chelsea,Burnley,2,3,A
3,EPL,2017-12-08 00:00:00,Crystal Palace,Huddersfield,0,3,A
4,EPL,2017-12-08 00:00:00,Everton,Stoke,1,0,H


In [5]:
myData.info()

<class 'pandas.DataFrame'>
RangeIndex: 2036 entries, 0 to 2035
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Div       2036 non-null   str   
 1   Date      2036 non-null   object
 2   HomeTeam  2036 non-null   str   
 3   AwayTeam  2036 non-null   str   
 4   FTHG      2036 non-null   int64 
 5   FTAG      2036 non-null   int64 
 6   FTR       2036 non-null   str   
dtypes: int64(2), object(1), str(4)
memory usage: 111.5+ KB


In [6]:
myData["Div"].unique()
myData["FTR"].unique()

<StringArray>
['H', 'A', 'D']
Length: 3, dtype: str

In [7]:
eplData = myData[myData["Div"] == "EPL"]
eplData['count'] = 1
eplData['hwin'] = np.where(eplData['FTR']=="H", 1, 0)
eplData['draw'] = np.where(eplData['FTR']=="D", 1, 0)
eplData['awin'] = np.where(eplData['FTR']=="A", 1, 0)
eplData.info()

<class 'pandas.DataFrame'>
RangeIndex: 380 entries, 0 to 379
Data columns (total 11 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Div       380 non-null    str   
 1   Date      380 non-null    object
 2   HomeTeam  380 non-null    str   
 3   AwayTeam  380 non-null    str   
 4   FTHG      380 non-null    int64 
 5   FTAG      380 non-null    int64 
 6   FTR       380 non-null    str   
 7   count     380 non-null    int64 
 8   hwin      380 non-null    int64 
 9   draw      380 non-null    int64 
 10  awin      380 non-null    int64 
dtypes: int64(6), object(1), str(4)
memory usage: 32.8+ KB


In [8]:
EPLHomeData = eplData.groupby("HomeTeam")[['count', "FTHG", "FTAG", 'hwin', 'draw', 'awin']].sum().reset_index()
EPLHomeData.rename(columns = {"HomeTeam": "Team",
                              "count": "Games",
                               "FTHG": "Goals Scored",
                                "FTAG": "Goals Conceded",
                                 "awin": "L",
                                  "hwin": "W",
                                   "draw": "D" },
                                inplace=True)
EPLHomeData

,Team,Games,Goals Scored,Goals Conceded,W,D,L
0,Arsenal,19,54,20,15,2,2
1,Bournemouth,19,26,30,7,5,7
2,Brighton,19,24,25,7,8,4
3,Burnley,19,16,17,7,5,7
4,Chelsea,19,30,16,11,4,4
5,Crystal Palace,19,29,27,7,5,7
6,Everton,19,28,22,10,4,5
7,Huddersfield,19,16,25,6,5,8
8,Leicester,19,25,22,7,6,6
9,Liverpool,19,45,10,12,7,0


In [9]:
EPLAwayData = eplData.groupby("AwayTeam")[['count', "FTHG", "FTAG", 'hwin', 'draw', 'awin']].sum().reset_index()
EPLAwayData.rename(columns = {"AwayTeam": "Team",
                              "count": "Games",
                               "FTHG": "Goals Conceded",
                                "FTAG": "Goals Scored",
                                 "awin": "W",
                                  "hwin": "L",
                                   "draw": "D"  },
                                inplace=True)
EPLAwayData


,Team,Games,Goals Conceded,Goals Scored,L,D,W
0,Arsenal,19,31,20,11,4,4
1,Bournemouth,19,31,19,9,6,4
2,Brighton,19,29,10,12,5,2
3,Burnley,19,22,20,5,7,7
4,Chelsea,19,22,32,6,3,10
5,Crystal Palace,19,28,16,9,6,4
6,Everton,19,36,16,10,6,3
7,Huddersfield,19,33,12,11,5,3
8,Leicester,19,38,31,9,5,5
9,Liverpool,19,28,39,5,5,9


In [10]:
EPLMerged = pd.merge(EPLHomeData, EPLAwayData, on="Team")
EPLMerged

,Team,Games_x,Goals Scored_x,Goals Conceded_x,W_x,D_x,L_x,Games_y,Goals Conceded_y,Goals Scored_y,L_y,D_y,W_y
0,Arsenal,19,54,20,15,2,2,19,31,20,11,4,4
1,Bournemouth,19,26,30,7,5,7,19,31,19,9,6,4
2,Brighton,19,24,25,7,8,4,19,29,10,12,5,2
3,Burnley,19,16,17,7,5,7,19,22,20,5,7,7
4,Chelsea,19,30,16,11,4,4,19,22,32,6,3,10
5,Crystal Palace,19,29,27,7,5,7,19,28,16,9,6,4
6,Everton,19,28,22,10,4,5,19,36,16,10,6,3
7,Huddersfield,19,16,25,6,5,8,19,33,12,11,5,3
8,Leicester,19,25,22,7,6,6,19,38,31,9,5,5
9,Liverpool,19,45,10,12,7,0,19,28,39,5,5,9


In [11]:
EPLMerged["GP"] = EPLMerged['Games_x'] + EPLMerged['Games_y']
EPLMerged["W"] = EPLMerged['W_x'] + EPLMerged['W_y']
EPLMerged["D"] = EPLMerged['D_x'] + EPLMerged['D_y']
EPLMerged["L"] = EPLMerged['L_x'] + EPLMerged['L_y']
EPLMerged["GF"] = EPLMerged['Goals Scored_x'] + EPLMerged['Goals Scored_y']
EPLMerged["GA"] = EPLMerged['Goals Conceded_x'] + EPLMerged['Goals Conceded_y']
EPLMerged["GD"] = EPLMerged['GF'] - EPLMerged['GA']
EPLMerged["Pts"] = EPLMerged['W'] * 3 + EPLMerged['D']

EPLMerged = EPLMerged[["Team","GP", "W", "D", "L", "GF", "GA", "GD", "Pts"]]
EPLMerged

,Team,GP,W,D,L,GF,GA,GD,Pts
0,Arsenal,38,19,6,13,74,51,23,63
1,Bournemouth,38,11,11,16,45,61,-16,44
2,Brighton,38,9,13,16,34,54,-20,40
3,Burnley,38,14,12,12,36,39,-3,54
4,Chelsea,38,21,7,10,62,38,24,70
5,Crystal Palace,38,11,11,16,45,55,-10,44
6,Everton,38,13,10,15,44,58,-14,49
7,Huddersfield,38,9,10,19,28,58,-30,37
8,Leicester,38,12,11,15,56,60,-4,47
9,Liverpool,38,21,12,5,84,38,46,75


In [12]:
EPLMerged.sort_values(by="Pts", inplace=True, ascending=False)
EPLMerged

,Team,GP,W,D,L,GF,GA,GD,Pts
10,Man City,38,32,4,2,106,27,79,100
11,Man United,38,25,6,7,68,28,40,81
16,Tottenham,38,23,8,7,74,36,38,77
9,Liverpool,38,21,12,5,84,38,46,75
4,Chelsea,38,21,7,10,62,38,24,70
0,Arsenal,38,19,6,13,74,51,23,63
3,Burnley,38,14,12,12,36,39,-3,54
6,Everton,38,13,10,15,44,58,-14,49
8,Leicester,38,12,11,15,56,60,-4,47
1,Bournemouth,38,11,11,16,45,61,-16,44
